In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import multiprocessing
import os
import gdown

In [ ]:
# 1. zip 파일의 ID 입력
zip_file_id = 'YOUR_DRIVE_FILE_ID'
output_filename = 'data.zip'

# 2. zip 파일 다운로드
url = f'https://drive.google.com/uc?id={zip_file_id}'
gdown.download(url, output_filename, quiet=False)

# 3. 압축 풀기 (리눅스 명령어를 사용합니다)
# -o: 덮어쓰기 허용, -d: 압축 풀 폴더 지정
!unzip -o data.zip

# 4. 확인
print("파일 목록:", os.listdir())

train = pd.read_csv('data/train.csv', encoding='utf-8-sig')
test = pd.read_csv('data/test.csv', encoding='utf-8-sig')
submission = pd.read_csv('data/sample_submission.csv', encoding='utf-8-sig')

In [ ]:
# ==========================================
# 1. 자모 유틸리티 & 정렬(Alignment) 함수
# ==========================================
BASE_CODE = 0xAC00
CHOSUNG  = 21 * 28
JUNGSUNG = 28
CHOSUNG_LIST = ['ㄱ','ㄲ','ㄴ','ㄷ','ㄸ','ㄹ','ㅁ','ㅂ','ㅃ','ㅅ','ㅆ','ㅇ','ㅈ','ㅉ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']
JUNGSUNG_LIST = ['ㅏ','ㅐ','ㅑ','ㅒ','ㅓ','ㅔ','ㅕ','ㅖ','ㅗ','ㅘ','ㅙ','ㅚ','ㅛ','ㅜ','ㅝ','ㅞ','ㅟ','ㅠ','ㅡ','ㅢ','ㅣ']
JONGSUNG_LIST = ['', 'ㄱ','ㄲ','ㄳ','ㄴ','ㄵ','ㄶ','ㄷ','ㄹ','ㄺ','ㄻ','ㄼ','ㄽ','ㄾ','ㄿ','ㅀ','ㅁ','ㅂ','ㅄ','ㅅ','ㅆ','ㅇ','ㅈ','ㅊ','ㅋ','ㅌ','ㅍ','ㅎ']

def to_jamo_list(text):
    result = []
    for ch in text:
        code = ord(ch) - BASE_CODE
        if code < 0 or code > (0xD7A3 - 0xAC00):
            result.append(ch)
            continue
        cho  = code // CHOSUNG
        jung = (code % CHOSUNG) // JUNGSUNG
        jong = (code % CHOSUNG) % JUNGSUNG
        result.extend([CHOSUNG_LIST[cho], JUNGSUNG_LIST[jung], JONGSUNG_LIST[jong]])
    return result

def from_jamo_list(jamo_list):
    result = []
    i = 0
    while i < len(jamo_list) - 2:
        cho, jung, jong = jamo_list[i], jamo_list[i+1], jamo_list[i+2]
        if cho in CHOSUNG_LIST and jung in JUNGSUNG_LIST and jong in JONGSUNG_LIST:
            cho_i = CHOSUNG_LIST.index(cho)
            jung_i = JUNGSUNG_LIST.index(jung)
            jong_i = JONGSUNG_LIST.index(jong)
            code = BASE_CODE + (cho_i * CHOSUNG) + (jung_i * JUNGSUNG) + jong_i
            result.append(chr(code))
            i += 3
        else:
            result.append(cho)
            i += 1
    return "".join(result)

def align_jamo_list(src_list, tgt_list):
    """
    두 리스트의 길이를 맞추기 위해 최적의 정렬을 수행 (Levenshtein Distance 기반)
    Returns: [(src_char, tgt_char), ...]
    """
    n, m = len(src_list), len(tgt_list)
    dp = [[0]*(m+1) for _ in range(n+1)]
    
    for i in range(1, n+1): dp[i][0] = i
    for j in range(1, m+1): dp[0][j] = j
    
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if src_list[i-1] == tgt_list[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
            
    alignment = []
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and dp[i][j] == dp[i-1][j] + 1:
            # 삭제됨 (Input에는 있는데 Output에는 없음 -> Output은 <DEL>)
            alignment.append((src_list[i-1], '<DEL>'))
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            # 삽입됨 (Input엔 없는데 Output엔 있음)
            # 분류 모델 특성상 Input이 없는 위치를 예측하긴 어려우므로 무시(Skip)
            alignment.append((None, tgt_list[j-1]))
            j -= 1
        else:
            # 매칭 또는 치환
            alignment.append((src_list[i-1], tgt_list[j-1]))
            i -= 1
            j -= 1
            
    return alignment[::-1]

# ==========================================
# 2. 데이터셋 생성 (Alignment 적용)
# ==========================================
def create_dataset_aligned(df, window_size=7):
    data_X = []
    data_y = []
    PAD = '<PAD>'
    
    print("Generating Aligned Dataset...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        in_jamos = to_jamo_list(row['input'])
        out_jamos = to_jamo_list(row['output'])
        
        # 1. 정렬 수행 (길이가 달라도 매칭)
        alignment = align_jamo_list(in_jamos, out_jamos)
        
        # 2. 윈도우 생성을 위한 패딩된 입력
        padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
        
        # 3. 데이터셋 추출
        # alignment를 순회하되, src_char가 None인 경우(삽입)는 건너뜀
        # (입력 문자가 존재해야 분류를 할 수 있으므로)
        current_input_idx = 0
        
        for src_char, tgt_char in alignment:
            if src_char is None:
                continue # 삽입된 패턴은 현재 모델 구조상 학습 제외
            
            # 현재 입력 문자의 윈도우 추출
            window = padded_in[current_input_idx : current_input_idx + window_size]
            data_X.append(window)
            data_y.append(tgt_char) # tgt_char는 정답 문자이거나 '<DEL>'
            
            current_input_idx += 1
            
    return data_X, data_y

# ==========================================
# 3. 메인 파이프라인
# ==========================================

# 1) 데이터셋 생성 (Window Size 7로 증가)
WINDOW_SIZE = 7
X_raw, y_raw = create_dataset_aligned(train, window_size=WINDOW_SIZE)

print(f"Total Aligned Samples: {len(X_raw)}")

# 2) 인코딩 (Label Encoding)
all_tokens = set(np.array(X_raw).flatten()) | set(y_raw)
if '<DEL>' not in all_tokens: all_tokens.add('<DEL>')

token_to_id = {t: i for i, t in enumerate(sorted(list(all_tokens)))}
id_to_token = {i: t for t, i in token_to_id.items()}

def encode_features(X_list, token_map):
    encoded = []
    unknown_token = token_map.get('<PAD>', 0)
    for sample in X_list:
        encoded.append([token_map.get(t, unknown_token) for t in sample])
    return np.array(encoded)

print("Encoding Features...")
X_encoded = encode_features(X_raw, token_to_id).astype(np.int16)
y_encoded = np.array([token_to_id[t] for t in y_raw]).astype(np.int16)

# 3) 모델 학습
print("Training RandomForest Classifier...")
# n_estimators=100으로 증가, max_depth=25로 약간 상향
model = RandomForestClassifier(
    n_estimators=100,      
    n_jobs=1,              # 메모리 안전 모드
    max_depth=25,          
    min_samples_split=5,
    random_state=42, 
    verbose=1
)
model.fit(X_encoded, y_encoded)
print("Training Done!")

# ==========================================
# 4. 추론 및 저장
# ==========================================
def restore_text_aligned(text, model, token_map, inv_token_map, window_size=7):
    in_jamos = to_jamo_list(text)
    PAD = '<PAD>'
    padded_in = [PAD] * (window_size // 2) + in_jamos + [PAD] * (window_size // 2)
    
    windows = []
    for i in range(len(in_jamos)):
        win = padded_in[i : i + window_size]
        windows.append(win)
    
    X_test = encode_features(windows, token_map)
    y_pred = model.predict(X_test)
    
    restored_jamos = []
    for pid in y_pred:
        token = inv_token_map[pid]
        if token != '<DEL>': # <DEL>로 예측되면 해당 문자 삭제
            restored_jamos.append(token)
            
    return from_jamo_list(restored_jamos)

print("Restoring Test Data...")
outputs = []
for idx, row in tqdm(test.iterrows(), total=len(test)):
    restored = restore_text_aligned(row['input'], model, token_to_id, id_to_token, window_size=WINDOW_SIZE)
    outputs.append(restored)



In [ ]:
submission['output'] = outputs
submission.to_csv('submission_sklearn_v2.csv', index=False)
print("Done! Saved to submission_sklearn_v2.csv")